# 🚦 Day 3 — Evaluation Gate on RunPod

Load your Day 2 winning model, run it through the **golden-fact gate**, read the failures,
and **register it only if it passes**.

**The gate blocks a model unless ALL three conditions hold:**
1. **Absolute floor** — ≥ 80% of golden cases pass
2. **No regression** — ≥ the currently-deployed model (skipped on first run)
3. **No slice collapse** — no task category below 50%

> This is NOT deployment. The gate *loads* the model and asks it questions — it decides
> *whether* the model earns deployment (Day 4). Loading ≠ serving.

**Sections:**
1. Install (pinned, then restart kernel)
2. Environment check
3. Config — model + thresholds
4. Load the golden set
5. Load the model (from HF Hub)
6. Generate answers to every golden question
7. Score + apply the gate
8. Read the failures
9. See it BLOCK (sanity)
10. Register the model (only if approved)

## 1. Install dependencies (then RESTART THE KERNEL)

Same matched, pinned set as Day 2 — don't reinstall torch (it's matched to the pod's driver).
After it runs: **Kernel → Restart**, then skip this cell.

In [ ]:
import torch; print(torch.__version__)   # should be 2.5.1+cu121 from before

In [ ]:
%pip install  \
    "transformers==4.46.3" \
    "tokenizers==0.20.3" \
    "accelerate==1.1.1" \
    "huggingface_hub>=0.26.0" \
    "hf_transfer" \
    "pyyaml>=6.0" \
    "sentencepiece>=0.2.0"

In [ ]:
import torch, transformers, accelerate
print(torch.__version__, torch.cuda.is_available())

## 2. Environment check

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16:", torch.cuda.is_bf16_supported())

if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    DTYPE = torch.bfloat16
elif torch.cuda.is_available():
    DTYPE = torch.float16
else:
    DTYPE = torch.float32
print(">> dtype:", DTYPE)

## 3. Configuration — model + gate thresholds

Mirrors `configs/eval.yaml`. Point `MODEL` at your Day 2 winner on the Hub.

In [ ]:
CONFIG = {
    # the model to gate: your Day 2 merged winner (HF repo id or local path)
    "model": "vinmlops/domainbot-1.5b-rank32",     # <-- change to yours

    # champion to compare against; None on the first gate (no deployed model yet)
    "baseline_model": None,

    "system_prompt": ("You are DomainBot, a helpful and precise assistant. "
                      "Answer accurately and concisely, and say when you don't know."),
    "max_new_tokens": 200,
    "temperature": 0.0,        # deterministic — a gate must be reproducible

    # ---- the three gate conditions ----
    "min_pass_rate": 0.80,           # absolute floor
    "min_vs_baseline": 0.0,          # new >= baseline (no regression)
    "min_category_pass_rate": 0.50,  # no category may collapse
}
print("gating:", CONFIG["model"])

## 4. Load the golden set

Upload your `golden_set.jsonl` (from the repo, `data/golden/`) via the Jupyter file browser,
OR the cell writes a small built-in set so the notebook runs standalone.

Each case: a question, `must_contain` (≥1 must appear), `must_not_contain` (none may appear),
and a category. The `must_not_contain` bans are what make this a **regression suite**.

In [ ]:
import os, json

GOLDEN_PATH = "data/golden/golden_set.jsonl"

builtin = [
    {"question":"A train travels 120 km in 2 hours. What is its average speed?",
     "must_contain":["60"],"must_not_contain":["30","120 km/h"],"category":"cot"},
    {"question":"What is 15% of 200?","must_contain":["30"],
     "must_not_contain":["15","300"],"category":"cot"},
    {"question":"What is the capital of Japan?","must_contain":["tokyo"],
     "must_not_contain":["kyoto","osaka"],"category":"flan"},
    {"question":"What is the capital of France?","must_contain":["paris"],
     "must_not_contain":[],"category":"flan"},
    {"question":"At what temperature does water boil at sea level in Celsius?",
     "must_contain":["100"],"must_not_contain":["90","212"],"category":"flan"},
    {"question":"Rewrite in passive voice: 'The team finished the project.'",
     "must_contain":["was finished"],"must_not_contain":[],"category":"niv"},
    {"question":"Classify sentiment as positive or negative: 'This exceeded my expectations.'",
     "must_contain":["positive"],"must_not_contain":[],"category":"t0"},
    {"question":"What was my bank balance last Tuesday?",
     "must_contain":["don't have access","do not have access","can't","cannot"],
     "must_not_contain":[],"category":"honesty"},
    {"question":"What will the stock market do tomorrow?",
     "must_contain":["can't predict","cannot predict","don't know","no one"],
     "must_not_contain":[],"category":"honesty"},
]

if os.path.exists(GOLDEN_PATH):
    golden = [json.loads(l) for l in open(GOLDEN_PATH) if l.strip()]
    print(f"loaded {len(golden)} cases from {GOLDEN_PATH}")
else:
    golden = builtin
    print(f"using {len(golden)} built-in cases (upload golden_set.jsonl to use your own)")

from collections import Counter
print("categories:", dict(Counter(g["category"] for g in golden)))

## 5. Load the model from the HF Hub

This *loads* the model to ask it questions — it does not deploy it. If the repo is private,
log in first (cell will prompt).

In [ ]:
from huggingface_hub import login
# login()   # uncomment if your model repo is PRIVATE

from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(CONFIG["model"])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model"], torch_dtype=DTYPE, device_map="auto",
).eval()
print(">> loaded:", CONFIG["model"])

## 6. Generate an answer to every golden question

Temperature 0 = deterministic, so the gate gives the same verdict every run.

In [ ]:
import torch

def answer(question):
    msgs = [{"role":"system","content":CONFIG["system_prompt"]},
            {"role":"user","content":question}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=CONFIG["max_new_tokens"],
                             do_sample=False, pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()

answers = []
for i, g in enumerate(golden, 1):
    a = answer(g["question"])
    answers.append(a)
    print(f"[{i}/{len(golden)}] {g['question'][:50]}")
print(">> generated all answers")

## 7. Score + apply the gate

Pure logic (this is `src/evaluation/evaluate.py` inlined). A case passes iff a `must_contain`
pattern matches AND no `must_not_contain` pattern matches.

In [ ]:
import re
from collections import defaultdict

def judge(ans, must, mustnot):
    ok_must = any(re.search(p, ans, re.I) for p in must) if must else True
    hits = [p for p in mustnot if re.search(p, ans, re.I)]
    return (ok_must and not hits), hits

cases, cat_pass, cat_total, passed = [], defaultdict(int), defaultdict(int), 0
for ans, g in zip(answers, golden):
    ok, hits = judge(ans, g.get("must_contain",[]), g.get("must_not_contain",[]))
    passed += ok
    cat_total[g["category"]] += 1; cat_pass[g["category"]] += ok
    cases.append({"q":g["question"],"a":ans,"cat":g["category"],"pass":ok,
                  "forbidden":hits,"expected":g.get("must_contain",[])})

n = len(golden)
pass_rate = passed / n
per_cat = {c:{"pass":cat_pass[c],"total":cat_total[c],"rate":cat_pass[c]/cat_total[c]}
           for c in sorted(cat_total)}

# ---- the three gate conditions ----
reasons, approved = [], True
if pass_rate < CONFIG["min_pass_rate"]:
    approved = False; reasons.append(f"pass_rate {pass_rate:.0%} < floor {CONFIG['min_pass_rate']:.0%}")
# baseline check skipped here (set baseline_model + evaluate it to enable)
for c, s in per_cat.items():
    if s["rate"] < CONFIG["min_category_pass_rate"]:
        approved = False; reasons.append(f"category '{c}' {s['rate']:.0%} < {CONFIG['min_category_pass_rate']:.0%}")
if approved:
    reasons.append("all gate conditions satisfied")

print(f"OVERALL: {passed}/{n} ({pass_rate:.0%})")
for c, s in per_cat.items():
    flag = "" if s["rate"] >= CONFIG["min_category_pass_rate"] else "  <-- LOW"
    print(f"  {c:10} {s['pass']}/{s['total']} ({s['rate']:.0%}){flag}")
print("\nGATE:", "APPROVED ✅" if approved else "BLOCKED ❌")
for r in reasons: print("  -", r)

# save report
with open("eval_report.json","w") as f:
    json.dump({"model":CONFIG["model"],"pass_rate":pass_rate,"approved":approved,
               "reasons":reasons,"per_category":per_cat,"cases":cases}, f, indent=2)
print(">> wrote eval_report.json")

## 8. Read the failures — not just the score

A blocked model is the gate doing its job. Read *why* each case failed.

> Expect some failures if your model has the Day 2 PII over-scrubbing bug (INC-013) — it may
> emit `<LOCATION>` placeholders. That's a real defect the gate correctly catches. Fix = correct
> the data, retrain, re-gate. **Never** edit the golden set to force a pass.

In [ ]:
fails = [c for c in cases if not c["pass"]]
if not fails:
    print("no failures — all cases passed 🎉")
for c in fails:
    print("FAIL:", c["q"][:60])
    print("  got     :", c["a"][:120])
    if c["forbidden"]:
        print("  banned  :", c["forbidden"])
    else:
        print("  expected:", c["expected"])
    print()

## 9. Sanity — see the gate BLOCK (do once)

A gate you've never seen block isn't a gate you can trust. Raise the floor to 0.99 and re-decide
on the same results — APPROVED should flip to BLOCKED.

In [ ]:
strict = 0.99
blocked = pass_rate < strict or any(s["rate"] < CONFIG["min_category_pass_rate"] for s in per_cat.values())
print(f"at floor {strict:.0%}: ", "BLOCKED ❌" if blocked else "APPROVED ✅",
      f"(actual pass_rate {pass_rate:.0%})")
print(">> proves the gate blocks when the bar is above the model's score")

## 10. Register the model — only if APPROVED

Tags the HF repo `v1.0.0` and writes a model card with the eval result. Skips if the gate blocked.

Needs an HF token with **write** access.

In [ ]:
if not approved:
    print("BLOCKED — not registering. Fix the model/data, retrain, re-gate.")
else:
    from huggingface_hub import login, create_tag, upload_file
    import io
    login()   # write token

    VERSION = "v1.0.0"
    try:
        create_tag(repo_id=CONFIG["model"], tag=VERSION, repo_type="model")
        print(f">> tagged {CONFIG['model']} @ {VERSION}")
    except Exception as e:
        print("tag note:", e)

    card = f"""---
license: apache-2.0
tags: [domainbot, qlora, gated]
---
# DomainBot — {VERSION}
Passed the golden-fact gate before registration.
- Golden pass rate: {pass_rate:.0%} ({passed}/{n})
- Gate: APPROVED
Versioning: MAJOR=new base/prompt, MINOR=new data, PATCH=hyperparams.
"""
    upload_file(path_or_fileobj=io.BytesIO(card.encode()), path_in_repo="README.md",
                repo_id=CONFIG["model"], repo_type="model",
                commit_message=f"register {VERSION}: gate passed")
    print(f">> registered {CONFIG['model']} @ {VERSION}")

In [ ]:
from huggingface_hub import list_repo_refs
print([t.name for t in list_repo_refs(CONFIG["model"]).tags])   # ['v1.0.0']

## ✅ Done

- Model evaluated against the golden set (loaded, not deployed)
- Gate verdict + per-category breakdown + failures read
- `eval_report.json` written (download it for your portfolio)
- Registered `v1.0.0` **only if it passed**

**If blocked:** that's a strong portfolio story — the gate caught a real defect. Fix the data
(INC-013), retrain (Day 2), re-gate. Never edit the golden set to force a pass.

**Terminate the pod** when done (a stopped pod still bills storage).

**Next — Day 4:** wrap the registered model in a vLLM + FastAPI service with the server-side
chat template, validation, streaming, and health endpoints. *That* is deployment.